# Notebook 02 — ETL Pipeline (Cleaning & Transformation)
**SuperStore Sales & Business Performance Analytics System**

This notebook demonstrates:
- Step-by-step data cleaning
- Derived metric creation
- Before vs. after comparison
- Saving cleaned data as Parquet

In [ ]:
import sys, os

# Detect Databricks
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IN_DATABRICKS:
    import subprocess
    repo_dir = '/tmp/superstore_repo'
    if not os.path.exists(repo_dir):
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/sachintulla/Databrick-Assignment.git', repo_dir],
            check=True, capture_output=True
        )
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    MOUNT_POINT  = '/mnt/superstore'
    PROJECT_ROOT = repo_dir
    print(f'Databricks: repo cloned → {repo_dir}')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    MOUNT_POINT = None
    print(f'Local: PROJECT_ROOT={PROJECT_ROOT}')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
print('Setup complete.')

## 1. Load Raw Data

In [ ]:
from src.ingestion.data_loader import DataLoader

loader = DataLoader()
if IN_DATABRICKS:
    raw_df = loader.load(f'/dbfs{MOUNT_POINT}/bronze/superstore_sales.csv')
else:
    raw_df = loader.load()

print(f'Raw shape: {raw_df.shape}')
print(f'Null counts:\n{raw_df.isnull().sum()[raw_df.isnull().sum() > 0]}')
print(f'Duplicate rows: {raw_df.duplicated().sum()}')


## 2. Apply Cleaning Pipeline

In [ ]:
from src.cleaning.data_cleaner import DataCleaner

cleaner = DataCleaner()
clean_df = cleaner.clean(raw_df)

print(f'Clean shape: {clean_df.shape}')
print(f'Rows removed: {len(raw_df) - len(clean_df):,}')
print(f'Null counts after cleaning:\n{clean_df.isnull().sum()[clean_df.isnull().sum() > 0]}')

## 3. Verify Derived Metrics

In [ ]:
derived_cols = ['profit_margin', 'discount_amount', 'shipping_days',
                'order_year', 'order_month', 'order_quarter', 'is_loss']
clean_df[derived_cols].describe().round(2)

In [ ]:
# Show loss-making rows
loss_df = clean_df[clean_df['is_loss']]
print(f'Loss-making line items: {len(loss_df):,} ({len(loss_df)/len(clean_df)*100:.1f}%)')
loss_df[['Product Name', 'Category', 'Sales', 'Profit', 'Discount', 'profit_margin']].head(10)

In [ ]:
# Profit margin distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(clean_df['profit_margin'].clip(-100, 100), bins=50,
         color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax1.set_title('Profit Margin Distribution')
ax1.set_xlabel('Profit Margin (%)')

ax2.hist(clean_df['shipping_days'].dropna(), bins=range(0, 15),
         color='coral', edgecolor='white', alpha=0.8)
ax2.set_title('Shipping Days Distribution')
ax2.set_xlabel('Shipping Days')

plt.suptitle('Derived Metrics Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Save Cleaned Data as Parquet

In [ ]:
saved_path = cleaner.save(clean_df)
print(f'Saved to: {saved_path}')

# Verify round-trip
verify_df = pd.read_parquet(saved_path)
print(f'Round-trip verification shape: {verify_df.shape}')
assert verify_df.shape == clean_df.shape, 'Shape mismatch after save/load!'
print('Round-trip verification PASSED!')

In [ ]:
# Final schema
print('=== Final Clean DataFrame Schema ===')
print(clean_df.dtypes)